In [2]:
import pandas as pd
import yaml

from internalizer import Internalizer
from internalizer.calculation_setup import *
import time

In [6]:
from internalizer.internalizer import CONFIG_NO_REMOVAL

In [7]:
EI_VERSION = "3.10"

mifpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative-w-FE_2025-08-06_22.14.16/lca/remind_runs/remind_SSP2-NPi-internalize-test-iterative-w-FE.mif"
gdxpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative-w-FE_2025-08-06_22.14.16/input.gdx"
pathway = "SSP2-NPi-internalize-test-iterative-w-FE"
monetization = 0.5
# monetization = {
#     "ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)" : 5.27,
#   #   "ReCiPe 2016 v1.03, midpoint (H) - ecotoxicity: terrestrial - terrestrial ecotoxicity potential (TETP)": 6.4e-05
# }
plca = True
calcCosts = False
aggTaxes = False
# years = [2020, 2030, 2040, 2050]
years = [2030]
yaml_file = "../internalizer/data/mappings/remind_internalization_setup_noRemoval.yaml"


# set up brightway project
bw_project = f"internalizer_ei_{EI_VERSION}"

In [8]:
IMPACT_CATEGORIES_MC = [
    "acidification",
    "climate change",
    "ecotoxicity",
    "eutrophication",
    "fossil resources",
    "human toxicity",
    "ionizing radiation",
    "land use",
    "metal/mineral resources",
    "ozone depletion",
    "particulate matter formation",
    "photochemical oxidant formation",
    "water use"
]

In [9]:
# initialize Internalizer
I = Internalizer(
    mifpath,
    "remind",
    pathway,
    EI_VERSION,
    bw_project,
    gdxpath,
    outputfolder = "lca_noRemovalv2"
)
print(I.scenario)

SSP2-NPi-internalize-test-iterative-w-FE


In [10]:

if plca:
    t0 = time.time()
    # add_ES_subcategories(mifpath)
    I.run_premise(years)
    t1 = time.time()
    print(f"Premise runs done in {t1-t0} seconds", "\n")

else:
    I.years = years

I.set_calculation_setup(yaml_file=yaml_file) # defaults to REMIND Internalization setup
print("Calculation setup set", "\n")

if calcCosts:
    t0 = time.time()
    I.calculate_costs(monetization, save_intermediate_results=True)
    t1 = time.time()
    print(f"Cost calculation done in {t1-t0} seconds", "\n")

if aggTaxes:
    t0 = time.time()
    I.load_costs()

    ics = IMPACT_CATEGORIES_MC
    if isinstance(monetization, dict):
        ics = list(monetization.keys())
    I.write_remind_input_files(
        2020,
        2030,
        ics
    )
    t1 = time.time()
    print(f"Tax recalculation done in {t1-t0} seconds", "\n")


- Extracting source database
- Extracting inventories
- Fetching IAM data
Found file: remind_SSP2-NPi-internalize-test-iterative-w-FE
Reading remind_SSP2-NPi-internalize-test-iterative-w-FE as CSV file
The following variables are missing from the IAM file: /p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative-w-FE_2025-08-06_22.14.16/lca/remind_runs
+----------------------------------+
|             Variable             |
+----------------------------------+
| Production|Industry|Cement w CCS |
+----------------------------------+
The following variables are missing from the IAM file: /p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative-w-FE_2025-08-06_22.14.16/lca/remind_runs
+----------------------------------+
|             Variable             |
+----------------------------------+
| Production|Industry|Cement w CCS |
+----------------------------------+
Done!


Processing scenarios for all sectors:   0%|     | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [27]:
I.cs.data["SE"]["base mapping"]

,REMIND index,dataset name,dataset reference product,dataset unit,share
0,ngcc,"electricity production, natural gas, combined ...","electricity, high voltage",kilowatt hour,1.0
1,ngccc,"electricity production, at natural gas-fired c...","electricity, high voltage",kilowatt hour,1.0
2,ngt,"electricity production, natural gas, conventio...","electricity, high voltage",kilowatt hour,1.0
3,gaschp,"heat and power co-generation, natural gas, com...","electricity, high voltage",kilowatt hour,regional
4,gaschp,"heat and power co-generation, natural gas, com...","heat, district or industrial, natural gas",megajoule,regional
...,...,...,...,...,...
82,h2turb,"electricity production, from hydrogen-fired on...","electricity, high voltage",kilowatt hour,1.0
83,h22ch4,"methane, from electrochemical methanation, wit...","methane, from electrochemical methanation",kilogram,0.5
84,h22ch4,"methane, from electrochemical methanation, wit...","methane, from electrochemical methanation",kilogram,0.5
85,MeOH,"diesel production, synthetic, Fischer Tropsch ...","diesel, synthetic",kilogram,0.5


In [28]:
# load raw costs and aggregate with mappings
I.cost_results = {}
for lvl in I.cs.levels:
    I.cost_results[lvl] = {}
    for year in I.years:
        # recalculate mapping
        mapping = I.cs.data[lvl].get(
            "regionalized mapping", I.cs.regionalize_dynamic_mapping(lvl, year)
        )
        fp = I.outdir + f"/{I.model}/{I.scenario}/{str(year)}/regionalized_costs_{lvl}.csv"
        regionalized_costs = pd.read_csv(fp)
        costs_agg = combine_shares_and_costs(mapping, regionalized_costs).melt(
            var_name="impact category", value_name="cost", ignore_index=False
        ).reset_index()
        I.cost_results[lvl][year] = costs_agg.set_index(
            ["REMIND index", "region", "impact category"]
        )["cost"].to_xarray()

In [29]:
mapping

,REMIND index,dataset name,dataset reference product,dataset unit,share,region
0,build - fehes,"market for heat, district or industrial, natur...","heat, district or industrial, natural gas",megajoule,0.143422,CAZ
1,build - fehes,"heat production, air-water heat pump 10kW","heat, air-water heat pump 10kW",megajoule,0.856578,CAZ
2,build - feels,"heat, residential, electric storage heater, us...","heat, from residential heating system",megajoule,0.165798,CAZ
3,build - fegas,"heat production, natural gas, at boiler conden...","heat, central or small-scale, natural gas",megajoule,1.000000,CAZ
4,build - feh2s,"heat, residential, by combustion of hydrogen u...","heat, from residential heating system",megajoule,NaN,CAZ
...,...,...,...,...,...,...
2419,trans - fedie,"diesel, synthetic, burned in passenger car",heat,megajoule,0.001848,USA
2420,trans - feelt,"electricity, used in battery electric motorcycle","electricity, low voltage",megajoule,0.009380,USA
2421,trans - fepet,"bioethanol, burned in motorcycle",heat,megajoule,0.000068,USA
2422,trans - fepet,"petrol, burned in motorcycle",heat,megajoule,0.001323,USA


In [30]:
regionalized_costs = regionalized_costs.pivot(
    index=["dataset name", "dataset reference product", "dataset unit", "region"],
    columns="impact category",
    values="cost"
)

In [31]:
regionalized_costs

impact category                                                                                   acidification  \
dataset name                                       dataset reference product dataset unit region                  
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ          0.000239   
                                                                                          CHA          0.000257   
                                                                                          EUR          0.000245   
                                                                                          IND          0.000241   
                                                                                          JPN          0.000263   
...                                                                                                         ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    OAS          0.000221   
                                                                                          REF          0.000221   
                                                                                          SSA          0.000221   
                                                                                          USA          0.000221   
                                                                                          World        0.000221   

impact category                                                                                   climate change  \
dataset name                                       dataset reference product dataset unit region                   
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ           0.001880   
                                                                                          CHA           0.002326   
                                                                                          EUR           0.001781   
                                                                                          IND           0.001926   
                                                                                          JPN           0.002016   
...                                                                                                          ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    OAS           0.002214   
                                                                                          REF           0.002214   
                                                                                          SSA           0.002214   
                                                                                          USA           0.002214   
                                                                                          World         0.002214   

impact category                                                                                   ecotoxicity  \
dataset name                                       dataset reference product dataset unit region                
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ        0.000840   
                                                                                          CHA        0.001096   
                                                                                          EUR        0.000812   
                                                                                          IND        0.000880   
                                                                                          JPN        0.001059   
...                                                                                                       ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    O

In [32]:
mapping = mapping.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])

In [ ]:
old_idx = mapping.index
new_idx = []
for idx in old_idx:
    if idx in regionalized_costs.index:
        new_idx.append(idx)
    else:
        new_idx.append((idx[0], idx[1], idx[2], "World"))

In [ ]:
combined = pd.DataFrame(
    regionalized_costs.loc[new_idx].to_numpy(),
    index=old_idx,
    columns=regionalized_costs.columns
).mul(mapping["share"], axis=0)

In [ ]:
combined["REMIND index"] = mapping["REMIND index"]
combined

impact category                                                                                     ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)  \
dataset name                                       dataset reference product  dataset unit  region                                                                                                             
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ                                              0.000213                                                          
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ                                              0.000271                                                          
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ                                              0.000496                                                          
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ                                              0.000034                                                          
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ                                              0.000011                                                          
...                                                                                                                                               ...                                                          
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM                                              0.000727                                                          
                                                                                            MEA                                              0.000894                                                          
                                                                                            OAS                                              0.000844                                                          
                                                                                            SSA                                              0.000981                                                          
                                                                                            USA                                              0.000310                                                          

impact category                                                                                    REMIND index  
dataset name                                       dataset reference product  dataset unit  region               
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ            ngcc  
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ           ngccc  
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ             ngt  
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ           gastr  
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ           gastr  
...                                                                                                         ...  
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM         bioethl  
                                                                                            MEA         bioethl  
                                                                                            OAS         bioethl  
                                         

In [ ]:
combined.reset_index().groupby(["REMIND index", "region"])[regionalized_costs.columns].sum()

impact category      ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
REMIND index region                                                                                                          
MeOH         CAZ                                              0.002039                                                       
             CHA                                              0.002040                                                       
             EUR                                              0.002040                                                       
             IND                                              0.002040                                                       
             JPN                                              0.002040                                                       
...                                                                ...                                                       
windoff      NEU                                              0.000142                                                       
             OAS                                              0.000142                                                       
             REF                                              0.000142                                                       
             SSA                                              0.000142                                                       
             USA                                              0.000142                                                       

[624 rows x 1 columns]

In [ ]:
mapping.share

dataset name                                                                                                  dataset reference product   dataset unit   region
electricity production, natural gas, combined cycle power plant                                               electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, at natural gas-fired combined cycle power plant, post, pipeline 200km, storage 1000m  electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, natural gas, conventional power plant                                                 electricity, high voltage   kilowatt hour  CAZ       1.0
petroleum and gas production, onshore                                                                         natural gas, high pressure  cubic meter    CAZ       0.7
petroleum and gas production, offshore                                                                        natural gas, high pressure  cubic meter    CAZ       0.3
     

In [ ]:
combined.mul(mapping["share"], axis=0)

impact category                                                                                     ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
dataset name                                       dataset reference product  dataset unit  region                                                                                                          
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ                                              0.000213                                                       
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ                                              0.000271                                                       
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ                                              0.000496                                                       
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ                                              0.000034                                                       
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ                                              0.000011                                                       
...                                                                                                                                               ...                                                       
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM                                              0.000727                                                       
                                                                                            MEA                                              0.000894                                                       
                                                                                            OAS                                              0.000844                                                       
                                                                                            SSA                                              0.000981                                                       
                                                                                            USA                                              0.000310                                                       

[988 rows x 1 columns]

In [ ]:
global_idx = mapping.index.difference(regionalized_costs.index)
idx_frame = global_idx.to_frame().set_index(np.arange(len(global_idx)))
idx_frame["region"] = "World"

new_index = pd.MultiIndex.from_frame(idx_frame).union(mapping.index.intersection(regionalized_costs.index))
new_index

MultiIndex([(    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transe

In [ ]:
mapping["share"]

dataset name                                                                                                  dataset reference product   dataset unit   region
electricity production, natural gas, combined cycle power plant                                               electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, at natural gas-fired combined cycle power plant, post, pipeline 200km, storage 1000m  electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, natural gas, conventional power plant                                                 electricity, high voltage   kilowatt hour  CAZ       1.0
petroleum and gas production, onshore                                                                         natural gas, high pressure  cubic meter    CAZ       0.7
petroleum and gas production, offshore                                                                        natural gas, high pressure  cubic meter    CAZ       0.3
     

In [ ]:
regionalized_costs.loc[new_index]

impact category                                                                                           ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
dataset name                                       dataset reference product         dataset unit region                                                                                                          
biodiesel production, via transesterification, ... biodiesel, from palm oil          kilogram     IND                                              0.001617                                                       
                                                                                                  LAM                                              0.001534                                                       
                                                                                                  MEA                                              0.001690                                                       
                                                                                                  OAS                                              0.001581                                                       
                                                                                                  SSA                                              0.001646                                                       
...                                                                                                                                                     ...                                                       
wood pellet production                             wood pellet, measured as dry mass kilogram     World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       

[964 rows x 1 columns]

In [ ]:
Q = np.random.random((1, 100))
Imat = np.random.random((100, 50))

In [ ]:
(Q @ Imat).sum(axis=-1)

array([1388.29838872])

In [ ]:
import xarray as xr

In [ ]:
methods = list(monetization.keys())

In [ ]:
impacts = xr.DataArray(
    (Q @ Imat).sum(axis=-1),
    coords = {
        "LCIA method": methods
    }
)

In [ ]:
mfs = xr.DataArray(
    np.diag(list(monetization.values())),
    {
        "LCIA method": methods,
        "impact category": methods
    }
)

In [ ]:
(mfs * impacts).sum(dim="LCIA method").to_dataframe(name="cost").reset_index()

,impact category,cost
0,"ReCiPe 2016 v1.03, midpoint (H) - acidificatio...",7316.332509
